In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
input_directory = Path("/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/pca_lr_predictions_2008")
output_directory = Path("/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/daily_downscaled_2008")

output_directory.mkdir(parents=True, exist_ok=True)

input_files = {
    "humidity": input_directory / "pred_humidity_2008.nc",
    "longwave": input_directory / "pred_longwave_rad_2008.nc",
    "precipitation": input_directory / "pred_precipitation_2008.nc",
    "pressure": input_directory / "pred_pressure_2008.nc",
    "solar": input_directory / "pred_solar_2008.nc",
    "temperature": input_directory / "pred_temp_2008.nc",
    "u_wind": input_directory / "pred_u_wind_2008.nc",
    "v_wind": input_directory / "pred_v_wind_2008.nc",
}

target_variable_names = {
    "humidity": "qair",
    "longwave": "therm_rad",
    "precipitation": "precip",
    "pressure": "atmpres",
    "solar": "solar",
    "temperature": "tair",
    "u_wind": "u_wind",
    "v_wind": "v_wind",
}

notebook_github_url = "https://github.com/SalishSeaCast/analysis-dishika/blob/main/notebooks/PCA_new_grid/create_file.ipynb"

In [3]:
opened_datasets = {}
prediction_arrays = {}
source_variable_names = {}

for variable_label, file_path in input_files.items():
    current_dataset = xr.open_dataset(file_path)

    prediction_candidates = [
        variable_name
        for variable_name in current_dataset.data_vars
        if "time_counter" in current_dataset[variable_name].dims
        and "y" in current_dataset[variable_name].dims
        and "x" in current_dataset[variable_name].dims
    ]

    if len(prediction_candidates) != 1:
        raise ValueError(
            f"{file_path.name} contains "
            f"{len(prediction_candidates)} prediction candidates: "
            f"{prediction_candidates}"
        )

    source_variable_name = prediction_candidates[0]
    target_variable_name = target_variable_names[variable_label]

    prediction_array = current_dataset[source_variable_name].transpose(
        "time_counter",
        "y",
        "x",
    )

    opened_datasets[variable_label] = current_dataset
    prediction_arrays[variable_label] = prediction_array
    source_variable_names[variable_label] = source_variable_name

    print(
        f"{variable_label}: "
        f"{source_variable_name} -> {target_variable_name}, "
        f"shape = {prediction_array.shape}"
    )

humidity: qair -> qair, shape = (2904, 898, 398)
longwave: therm_rad -> therm_rad, shape = (2904, 898, 398)
precipitation: precip -> precip, shape = (2904, 898, 398)
pressure: atmpres -> atmpres, shape = (2904, 898, 398)
solar: solar -> solar, shape = (2904, 898, 398)
temperature: tair -> tair, shape = (2904, 898, 398)
u_wind: u_wind -> u_wind, shape = (2904, 898, 398)
v_wind: v_wind -> v_wind, shape = (2904, 898, 398)


In [4]:
reference_label = "temperature"
reference_dataset = opened_datasets[reference_label]
reference_array = prediction_arrays[reference_label]

reference_times = reference_dataset["time_counter"].values
reference_y_size = reference_dataset.sizes["y"]
reference_x_size = reference_dataset.sizes["x"]

reference_nav_lat = reference_dataset["nav_lat"].values
reference_nav_lon = reference_dataset["nav_lon"].values
reference_tmask = reference_dataset["tmask"].values

for variable_label, current_dataset in opened_datasets.items():
    current_array = prediction_arrays[variable_label]

    if current_dataset.sizes["y"] != reference_y_size:
        raise ValueError(
            f"Y dimension does not match for {variable_label}."
        )

    if current_dataset.sizes["x"] != reference_x_size:
        raise ValueError(
            f"X dimension does not match for {variable_label}."
        )

    if not np.array_equal(
        current_dataset["time_counter"].values,
        reference_times,
    ):
        raise ValueError(
            f"Timestamps do not match for {variable_label}."
        )

    if current_array.shape != reference_array.shape:
        raise ValueError(
            f"Prediction shape does not match for {variable_label}: "
            f"{current_array.shape} versus {reference_array.shape}"
        )

    if not np.allclose(
        current_dataset["nav_lat"].values,
        reference_nav_lat,
        equal_nan=True,
    ):
        raise ValueError(
            f"nav_lat does not match for {variable_label}."
        )

    if not np.allclose(
        current_dataset["nav_lon"].values,
        reference_nav_lon,
        equal_nan=True,
    ):
        raise ValueError(
            f"nav_lon does not match for {variable_label}."
        )

    if not np.array_equal(
        current_dataset["tmask"].values,
        reference_tmask,
    ):
        raise ValueError(
            f"tmask does not match for {variable_label}."
        )

print("All prediction files have matching timestamps and grids.")

All prediction files have matching timestamps and grids.


In [5]:
combined_data_variables = {}

for variable_label, prediction_array in prediction_arrays.items():
    target_variable_name = target_variable_names[variable_label]

    variable_attributes = dict(prediction_array.attrs)

    variable_attributes.pop("coordinates", None)
    variable_attributes.pop("_FillValue", None)
    variable_attributes.pop("missing_value", None)

    combined_data_variables[target_variable_name] = xr.DataArray(
        prediction_array.data,
        dims=(
            "time_counter",
            "y",
            "x",
        ),
        attrs=variable_attributes,
    )

In [6]:
nav_lat_attributes = dict(reference_dataset["nav_lat"].attrs)
nav_lon_attributes = dict(reference_dataset["nav_lon"].attrs)
tmask_attributes = dict(reference_dataset["tmask"].attrs)

nav_lat_attributes.pop("_FillValue", None)
nav_lon_attributes.pop("_FillValue", None)
tmask_attributes.pop("_FillValue", None)

combined_data_variables["nav_lat"] = xr.DataArray(
    reference_dataset["nav_lat"].data,
    dims=("y", "x"),
    attrs=nav_lat_attributes,
)

combined_data_variables["nav_lon"] = xr.DataArray(
    reference_dataset["nav_lon"].data,
    dims=("y", "x"),
    attrs=nav_lon_attributes,
)

combined_data_variables["tmask"] = xr.DataArray(
    reference_dataset["tmask"].data,
    dims=("y", "x"),
    attrs=tmask_attributes,
)

In [7]:
combined_dataset = xr.Dataset(
    data_vars=combined_data_variables,
    coords={
        "time_counter": (
            "time_counter",
            reference_times,
        ),
    },
)

combined_dataset["time_counter"].attrs.update(
    reference_dataset["time_counter"].attrs
)

combined_dataset["time_counter"].encoding = {}

In [8]:
combined_dataset.attrs["title"] = "PCA and multiple linear regression downscaled atmospheric forcing"
combined_dataset.attrs["data_split"] = "validation"
combined_dataset.attrs["year"] = 2008
combined_dataset.attrs["time_resolution"] = "3-hourly"
combined_dataset.attrs["notebook_github_url"] = notebook_github_url
combined_dataset.attrs["source_files"] = ", ".join(file_path.name for file_path in input_files.values())
combined_dataset.attrs["note"] = "Each output file contains all available downscaled atmospheric predictions for one date."

In [9]:
print(combined_dataset)

print("Coordinates:")
print(list(combined_dataset.coords))

print("Data variables:")
print(list(combined_dataset.data_vars))

if list(combined_dataset.coords) != ["time_counter"]:
    raise ValueError(
        "The combined dataset contains coordinates other than time_counter."
    )

if "nav_lat" not in combined_dataset.data_vars:
    raise ValueError(
        "nav_lat is not a data variable."
    )

if "nav_lon" not in combined_dataset.data_vars:
    raise ValueError(
        "nav_lon is not a data variable."
    )

<xarray.Dataset> Size: 33GB
Dimensions:       (time_counter: 2904, y: 898, x: 398)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 23kB 2008-01-01 ... 2008-12-3...
Dimensions without coordinates: y, x
Data variables:
    qair          (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    therm_rad     (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    precip        (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    atmpres       (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    solar         (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    tair          (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    u_wind        (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    v_wind        (time_counter, y, x) float32 4GB nan nan nan ... nan nan nan
    nav_lat       (y, x) float32 1MB 46.86 46.86 46.86 46.87 ... 51.1 51.1 51.1
    nav_lon       (y, x) float32 1MB -123.4 -123.4 -123.4 

In [10]:
combined_times = pd.DatetimeIndex(
    combined_dataset["time_counter"].values
)

if not np.all(combined_times.year == 2008):
    raise ValueError(
        "The combined dataset contains timestamps outside 2008."
    )

if combined_times.has_duplicates:
    duplicate_times = combined_times[
        combined_times.duplicated()
    ]

    raise ValueError(
        f"Duplicate timestamps were found: {duplicate_times}"
    )

if not combined_times.is_monotonic_increasing:
    raise ValueError(
        "The timestamps are not in increasing order."
    )

normalized_dates = combined_times.normalize()

timestamps_per_date = pd.Series(
    1,
    index=normalized_dates,
).groupby(level=0).sum()

expected_dates = pd.date_range(
    start="2008-01-01",
    end="2008-12-31",
    freq="D",
)

available_dates = pd.DatetimeIndex(
    timestamps_per_date.index
)

missing_dates = expected_dates.difference(
    available_dates
)

incomplete_dates = timestamps_per_date[
    timestamps_per_date != 8
]

print("Total timestamps:", len(combined_times))
print("Available dates:", len(available_dates))
print("Expected dates in 2008:", len(expected_dates))
print("Missing dates:", len(missing_dates))
print(
    "Dates without exactly 8 timestamps:",
    len(incomplete_dates),
)

if len(missing_dates) > 0:
    print("\nMissing dates:")
    print(missing_dates)

if len(incomplete_dates) > 0:
    print("\nDates without exactly 8 timestamps:")
    print(incomplete_dates)

Total timestamps: 2904
Available dates: 363
Expected dates in 2008: 366
Missing dates: 3
Dates without exactly 8 timestamps: 0

Missing dates:
DatetimeIndex(['2008-02-29', '2008-08-11', '2008-08-12'], dtype='datetime64[us]', freq=None)


In [11]:
prediction_variable_names = [
    "qair",
    "therm_rad",
    "precip",
    "atmpres",
    "solar",
    "tair",
    "u_wind",
    "v_wind",
]

required_data_variables = prediction_variable_names + [
    "nav_lat",
    "nav_lon",
    "tmask",
]

for variable_name in required_data_variables:
    if variable_name not in combined_dataset.data_vars:
        raise KeyError(
            f"{variable_name} is missing from the combined dataset."
        )

print("All required data variables are available.")

All required data variables are available.


In [12]:
print(combined_dataset["precip"].attrs)

{'units': 'kg/m^2/s', 'standard_name': 'precipitation_flux', 'long_name': 'PCA and multiple linear regression downscaled precipitation', 'grid': 'SalishSeaCast NEMO grid', 'prediction_method': 'PCA plus multiple linear regression'}


In [13]:
fill_value = np.float32(1e20)

for date_number, current_date in enumerate(
    available_dates,
    start=1,
):
    date_indices = np.flatnonzero(
        normalized_dates == current_date
    )

    if len(date_indices) != 8:
        raise ValueError(
            f"{current_date.date()} contains "
            f"{len(date_indices)} timestamps instead of 8."
        )

    daily_dataset = combined_dataset.isel(
        time_counter=date_indices
    )

    output_file_name = (
        f"downscaled_y{current_date.year:04d}"
        f"m{current_date.month:02d}"
        f"d{current_date.day:02d}.nc"
    )

    output_file_path = (
        output_directory / output_file_name
    )

    daily_dataset.attrs["title"] = (
        "PCA and multiple linear regression "
        "downscaled atmospheric forcing for "
        f"{current_date.strftime('%Y-%m-%d')}"
    )

    daily_dataset.attrs["date"] = (
        current_date.strftime("%Y-%m-%d")
    )

    daily_dataset.attrs["number_of_timestamps"] = int(
        len(date_indices)
    )

    daily_encoding = {}

    for variable_name in prediction_variable_names:
        daily_encoding[variable_name] = {
            "dtype": "float32",
            "_FillValue": fill_value,
            "zlib": True,
            "complevel": 4,
            "shuffle": True,
            "chunksizes": (
                len(date_indices),
                min(
                    200,
                    daily_dataset.sizes["y"],
                ),
                min(
                    200,
                    daily_dataset.sizes["x"],
                ),
            ),
        }

    daily_encoding["nav_lat"] = {
        "dtype": "float32",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
    }

    daily_encoding["nav_lon"] = {
        "dtype": "float32",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
    }

    daily_encoding["tmask"] = {
        "dtype": "int8",
        "_FillValue": None,
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
    }

    daily_encoding["time_counter"] = {
        "dtype": "float64",
        "units": "seconds since 1970-01-01 00:00:00",
        "calendar": "standard",
        "_FillValue": None,
    }

    daily_dataset.to_netcdf(
        output_file_path,
        mode="w",
        format="NETCDF4",
        encoding=daily_encoding,
    )

    print(
        f"Written {date_number}/{len(available_dates)}: "
        f"{output_file_name}",
        end="\r",
    )

print("\nDaily combined NetCDF files saved to:")
print(output_directory)

Written 363/363: downscaled_y2008m12d31.nc
Daily combined NetCDF files saved to:
/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/daily_downscaled_2008


In [14]:
generated_files = sorted(
    output_directory.glob(
        "downscaled_y2008m??d??.nc"
    )
)

print(
    "Number of generated files:",
    len(generated_files),
)

if len(generated_files) > 0:
    print(
        "First file:",
        generated_files[0],
    )

    print(
        "Last file:",
        generated_files[-1],
    )

Number of generated files: 363
First file: /ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/daily_downscaled_2008/downscaled_y2008m01d01.nc
Last file: /ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/daily_downscaled_2008/downscaled_y2008m12d31.nc


In [15]:
if len(generated_files) == 0:
    raise FileNotFoundError(
        "No daily NetCDF files were generated."
    )

sample_daily_file = generated_files[0]

sample_daily_dataset = xr.open_dataset(
    sample_daily_file
)

print(sample_daily_dataset)

print("\nFile:")
print(sample_daily_file.name)

print("\nDimensions:")
print(dict(sample_daily_dataset.sizes))

print("\nCoordinates:")
print(list(sample_daily_dataset.coords))

print("\nData variables:")
print(list(sample_daily_dataset.data_vars))

print("\nTimestamps:")
print(
    sample_daily_dataset[
        "time_counter"
    ].values
)

if list(sample_daily_dataset.coords) != [
    "time_counter"
]:
    raise ValueError(
        "The daily file contains coordinates "
        "other than time_counter."
    )

if "nav_lat" not in sample_daily_dataset.data_vars:
    raise ValueError(
        "nav_lat is not stored as a data variable."
    )

if "nav_lon" not in sample_daily_dataset.data_vars:
    raise ValueError(
        "nav_lon is not stored as a data variable."
    )

sample_daily_dataset.close()

<xarray.Dataset> Size: 95MB
Dimensions:       (time_counter: 8, y: 898, x: 398)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 64B 2008-01-01 ... 2008-01-01...
Dimensions without coordinates: y, x
Data variables:
    qair          (time_counter, y, x) float32 11MB ...
    therm_rad     (time_counter, y, x) float32 11MB ...
    precip        (time_counter, y, x) float32 11MB ...
    atmpres       (time_counter, y, x) float32 11MB ...
    solar         (time_counter, y, x) float32 11MB ...
    tair          (time_counter, y, x) float32 11MB ...
    u_wind        (time_counter, y, x) float32 11MB ...
    v_wind        (time_counter, y, x) float32 11MB ...
    nav_lat       (y, x) float32 1MB ...
    nav_lon       (y, x) float32 1MB ...
    tmask         (y, x) int8 357kB ...
Attributes:
    title:                 PCA and multiple linear regression downscaled atmo...
    data_split:            validation
    year:                  2008
    time_resolution:       3-hourly
    

In [16]:
sample_daily_dataset = xr.open_dataset(
    generated_files[0]
)

for variable_name in prediction_variable_names:
    print(
        variable_name,
        sample_daily_dataset[variable_name].dims,
        sample_daily_dataset[variable_name].shape,
    )

print(
    "nav_lat:",
    sample_daily_dataset["nav_lat"].dims,
    sample_daily_dataset["nav_lat"].shape,
)

print(
    "nav_lon:",
    sample_daily_dataset["nav_lon"].dims,
    sample_daily_dataset["nav_lon"].shape,
)

print(
    "tmask:",
    sample_daily_dataset["tmask"].dims,
    sample_daily_dataset["tmask"].shape,
)

sample_daily_dataset.close()

qair ('time_counter', 'y', 'x') (8, 898, 398)
therm_rad ('time_counter', 'y', 'x') (8, 898, 398)
precip ('time_counter', 'y', 'x') (8, 898, 398)
atmpres ('time_counter', 'y', 'x') (8, 898, 398)
solar ('time_counter', 'y', 'x') (8, 898, 398)
tair ('time_counter', 'y', 'x') (8, 898, 398)
u_wind ('time_counter', 'y', 'x') (8, 898, 398)
v_wind ('time_counter', 'y', 'x') (8, 898, 398)
nav_lat: ('y', 'x') (898, 398)
nav_lon: ('y', 'x') (898, 398)
tmask: ('y', 'x') (898, 398)


In [17]:
combined_dataset.close()

for current_dataset in opened_datasets.values():
    current_dataset.close()

print("All source datasets have been closed.")

All source datasets have been closed.
